#### Creating Dataframe object for the jobs that need to be scheduled

In [3]:
import pandas as pd

# Storing all job data details
jobs_data = {
    "jobs": ["A", "B", "C", "D", "E"],
    "arrival_time": [0, 1, 2, 3, 4],
    "processing_time": [11, 29, 31, 1, 2],
    "due_date": [61, 45, 31, 33, 32]
}

# Convert the raw job data into a DataFrame for scheduling.
df = pd.DataFrame(jobs_data)
df

,jobs,arrival_time,processing_time,due_date
0,A,0,11,61
1,B,1,29,45
2,C,2,31,31
3,D,3,1,33
4,E,4,2,32


#### Shortest Processing Time Dynamic Job Scheduling

In [4]:
# Sort by arrival time so the dynamic queue can be evaluated chronologically.
df_sorted = df.sort_values(by='arrival_time')
df_sorted

,jobs,arrival_time,processing_time,due_date
0,A,0,11,61
1,B,1,29,45
2,C,2,31,31
3,D,3,1,33
4,E,4,2,32


In [5]:
# Track which jobs are complete and store their performance metrics.
df_sorted["is_processed"] = False
df_sorted["finish_time"] = None
df_sorted["flow_time"] = None
df_sorted["tardiness"] = None

current_time = 0

# Iterating over the jobs till not all jobs are processed
while not df_sorted["is_processed"].all():

    # Checking all the available jobs
    available_jobs = df_sorted[(df_sorted["is_processed"] == False) & (df_sorted["arrival_time"] <= current_time)]

    # If no job is available, move time to the next job arrival
    if available_jobs.empty:
        next_arrival_time = df_sorted.loc[df_sorted["is_processed"] == False, "arrival_time"].min()
        current_time = next_arrival_time
        continue

    # Finding job with the shortest processing time
    shortest_processing_index = available_jobs["processing_time"].idxmin()

    # Updating finish time
    finish_time = current_time + df_sorted.loc[shortest_processing_index, "processing_time"]

    # Updating data in the dataframe for the calculated values
    df_sorted.loc[shortest_processing_index, "is_processed"] = True
    df_sorted.loc[shortest_processing_index, "finish_time"] = finish_time
    df_sorted.loc[shortest_processing_index, "flow_time"] = finish_time - df_sorted.loc[shortest_processing_index, "arrival_time"]
    df_sorted.loc[shortest_processing_index, "tardiness"] = max(finish_time - df_sorted.loc[shortest_processing_index, "due_date"], 0)
    
    # Updating current time with the finish time of the current job
    current_time = finish_time

In [6]:
df_sorted

,jobs,arrival_time,processing_time,due_date,is_processed,finish_time,flow_time,tardiness
0,A,0,11,61,True,11,11,0
1,B,1,29,45,True,43,42,0
2,C,2,31,31,True,74,72,43
3,D,3,1,33,True,12,9,0
4,E,4,2,32,True,14,10,0


In [7]:
# Calculate average tardiness and flow time for the dynamic SPT schedule.
avg_tardiness = df_sorted['tardiness'].mean()
avg_flow_time = df_sorted['flow_time'].mean()

In [8]:
print(f"Avg Tardiness: {avg_tardiness}, Avg Flow time: {avg_flow_time}")

Avg Tardiness: 8.6, Avg Flow time: 28.8
